# 3 — Persistence Baseline

**Purpose:** establish the forecasting error produced by assuming headcount stays unchanged for the next 30 minutes. The two regression models must be compared against this on the same eligible examples.

This notebook uses only the **training and validation CSVs** produced by Notebook 02. It does not open `test.csv`, fit parameters, change the data splits, or select a regression model.

A forecast made at time $t$ is
$$\hat y_{t+30\mathrm{min}}=\text{latest accepted headcount at }t.$$
The "current" and target measurements each follow Notebook 02's five-minute freshness rule. The target is one sensor count near the future instant, not the mean count over the next 30 minutes.

**Files needed:** the prepared data folder plus `src/__init__.py` and `src/evaluation.py` from this package. Use the project `.venv` kernel and run from top to bottom.

## 1 — Import the shared source functions

`src` is just a folder containing reusable Python code. Instead of copying slightly different error formulas into three notebooks, all three can import the same tested functions.

The path check assumes this notebook runs from `notebooks/`, as Notebook 02 does. Adding the project root to Python's search path makes the neighbouring `src` folder importable; it does not install another package.

In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import platform
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path("..").resolve()
assert (project_root / "src" / "evaluation.py").is_file(), (
    "Expected src/evaluation.py in the project root. Check file placement and working folder."
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation import persistence_predict, regression_metrics

print("Project root:", project_root)
print("Python:", platform.python_version(), "| pandas:", pd.__version__)

## 2 — Load the fixed training and validation examples

Use the files produced by Notebook 02. We do not resample, shuffle or remove additional rows. Even though persistence needs only the current count, it is evaluated on the complete five-input examples that linear regression and SVR will use.

The checks below verify the saved counts, row order, 30-minute horizon and partition boundaries. `feature_columns` explicitly lists allowed model inputs; target counts and target observation times are not predictors.

`preparation_settings.json` supplies the already-agreed settings. No final-test CSV is loaded.

In [ ]:
space_prefix = "227f1068"
data_folder = project_root / "data" / "prepared" / space_prefix
with open(data_folder / "preparation_settings.json", encoding="utf-8") as file:
    settings = json.load(file)

feature_columns = settings["feature_columns"]
target_column = settings["target_column"]
assert target_column == "target_headcount"
assert target_column not in feature_columns
assert settings["space_id"].startswith(space_prefix)
assert settings["horizon_minutes"] == 30

train = pd.read_csv(data_folder / "train.csv")
validation = pd.read_csv(data_folder / "validation.csv")
datasets = {"train": train, "validation": validation}
periods = {
    "train": (settings["train_start"], settings["validation_start"]),
    "validation": (settings["validation_start"], settings["test_start"])
}

for name, dataset in datasets.items():
    dataset["forecast_time"] = pd.to_datetime(dataset["forecast_time"], utc=True)
    dataset["target_time"] = pd.to_datetime(dataset["target_time"], utc=True)
    period_start, period_end = periods[name]
    assert len(dataset) == settings["split_counts"][name]
    assert len(dataset) > 0
    assert dataset["forecast_time"].is_unique
    assert dataset["forecast_time"].is_monotonic_increasing
    assert dataset[feature_columns + [target_column]].notna().all().all()
    assert (dataset["forecast_time"] >= pd.Timestamp(period_start)).all()
    assert (dataset["target_time"] < pd.Timestamp(period_end)).all()
    assert ((dataset["target_time"] - dataset["forecast_time"]) == pd.Timedelta(minutes=30)).all()
    print(f"{name}: {len(dataset):,} examples checked")

print("The final-test CSV has not been opened.")

## 3 — Understand and check the error calculations

Let $e_i=y_i-\hat y_i$ (recorded target minus prediction).

$$\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|e_i|,
\qquad \mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}e_i^2}.$$

Both are in **people**, not percentages. MAE treats each additional person of error equally; RMSE gives larger errors extra influence through squaring. We report both for every model rather than selecting whichever makes a model look best. Neither metric proves HVAC suitability or energy savings.

For this small made-up example, residuals are `[1, -2, 2]`: MAE is $5/3$ and RMSE is $\sqrt{3}$. The shared function should reproduce these hand calculations. `np.isclose` allows tiny floating-point rounding differences.

In [ ]:
example_observed = [3, 5, 10]
example_predicted = [2, 7, 8]
example_scores = regression_metrics(example_observed, example_predicted)

assert np.isclose(example_scores["mae"], 5 / 3)
assert np.isclose(example_scores["rmse"], np.sqrt(3))
print("Hand-calculated example passed:", example_scores)

## 4 — Generate persistence forecasts and calculate errors

For every row, the prediction is simply `headcount_now`. There is **no fitting stage** and no learned parameter.

The table reports both partitions so future model notebooks can use the same reporting format. A difference between persistence's training-period and validation-period errors reflects the periods' data, not overfitting by persistence: it has learned nothing.

Residuals remain attached to their forecast and target timestamps for later comparison. Positive residuals mean the forecast was too low; negative residuals mean it was too high. No predictions are rounded or clipped here.

In [ ]:
result_rows = []
predictions_by_split = {}

for name, dataset in datasets.items():
    predicted = persistence_predict(dataset["headcount_now"].to_numpy())
    observed = dataset[target_column].to_numpy()
    scores = regression_metrics(observed, predicted)

    result_rows.append({
        "model": "persistence",
        "split": name,
        "examples": len(dataset),
        "mae_people": scores["mae"],
        "rmse_people": scores["rmse"]
    })

    predictions = dataset[["forecast_time", "target_time", "target_headcount_observed_at"]].copy()
    predictions["observed_headcount"] = observed
    predictions["predicted_headcount"] = predicted
    predictions["residual_people"] = observed - predicted
    predictions_by_split[name] = predictions

metrics = pd.DataFrame(result_rows)
display(metrics.round(4))

## 5 — Measure the cost of making predictions

We time only the shared prediction function on the full validation input array, excluding CSV loading, metric calculations and plots. The function includes its input checks and returns a separate output array.

The baseline is very fast, so each measurement repeats the same batch 100 times. Dividing by 100 gives seconds per batch; the median of five measurements reduces timing noise. This repeats a computation, **not training**. Runtime depends on the computer and is not an accuracy score.

Later comparisons should use a consistent timing scope and the same computer. This is an initial timing measurement, not the complete project profiling analysis.

In [ ]:
validation_current = validation["headcount_now"].to_numpy()
persistence_predict(validation_current)

batch_times = []
for repeat in range(5):
    started = perf_counter()
    for batch in range(100):
        persistence_predict(validation_current)
    seconds_per_batch = (perf_counter() - started) / 100
    batch_times.append(seconds_per_batch)

median_seconds = float(np.median(batch_times))
timing = pd.DataFrame([{
    "model": "persistence",
    "split": "validation",
    "examples_per_batch": len(validation),
    "median_batch_seconds": median_seconds,
    "microseconds_per_example": median_seconds / len(validation) * 1_000_000,
    "repeats": 5,
    "batches_per_repeat": 100
}])
display(timing)

## 6 — Inspect a fixed validation window

Plot the first calendar week of the agreed validation period, selected by date rather than by which predictions look best. The horizontal axis is the **target time**, so each forecast is shown alongside its future reference observation.

Points are not joined across missing periods. Blank areas mean no eligible evaluated examples, not zero occupancy. The table below the plot reports residual direction across the entire validation set, not only the plotted week.

This is a diagnostic illustration, not independent proof of forecast reliability.

In [ ]:
validation_predictions = predictions_by_split["validation"]
week_start = pd.Timestamp(settings["validation_start"])
week_end = week_start + pd.Timedelta(days=7)
plot_data = validation_predictions[
    (validation_predictions["target_time"] >= week_start)
    & (validation_predictions["target_time"] < week_end)
]
plot_times = plot_data["target_time"].dt.tz_convert("Australia/Melbourne")

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(plot_times, plot_data["observed_headcount"], s=16, label="Recorded target")
ax.scatter(plot_times, plot_data["predicted_headcount"], s=18, marker="x", label="Persistence forecast")
ax.set_title("Persistence: first week of validation (eligible examples only)")
ax.set_xlabel("Target time (Melbourne)")
ax.set_ylabel("Headcount (people)")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

residuals = validation_predictions["residual_people"]
direction_summary = pd.DataFrame([{
    "mean_residual_people": residuals.mean(),
    "underprediction_percent": 100 * (residuals > 0).mean(),
    "exact_prediction_percent": 100 * (residuals == 0).mean(),
    "overprediction_percent": 100 * (residuals < 0).mean()
}])
display(direction_summary.round(3))

## 7 — Save the baseline evidence

Results go to `outputs/227f1068/persistence/`, keeping generated results separate from source code. Re-running this cell replaces these baseline outputs only. The project's existing `outputs/` exclusion should keep them out of Git; reviewed summary results will later be included in the report.

The JSON records versions, settings and fingerprints of the two input CSVs. A fingerprint is a content identifier: it helps later notebooks check they used the same prepared files, not a different dataset that happens to have the same number of rows. Nothing here opens the final-test CSV.

In [ ]:
output_folder = project_root / "outputs" / space_prefix / "persistence"
output_folder.mkdir(parents=True, exist_ok=True)

metrics.to_csv(output_folder / "metrics.csv", index=False)
timing.to_csv(output_folder / "timing.csv", index=False)
direction_summary.to_csv(output_folder / "validation_error_direction.csv", index=False)
for name, predictions in predictions_by_split.items():
    predictions.to_csv(output_folder / f"{name}_predictions.csv", index=False)
fig.savefig(output_folder / "validation_first_week.png", dpi=160, bbox_inches="tight")

fingerprints = {}
for name in datasets:
    fingerprints[name] = hashlib.sha256((data_folder / f"{name}.csv").read_bytes()).hexdigest()
run_information = {
    "model": "persistence",
    "rule": "Forecast equals the latest accepted current headcount; no fitting",
    "space_id": settings["space_id"],
    "horizon_minutes": settings["horizon_minutes"],
    "evaluated_partitions": list(datasets),
    "test_data_used": False,
    "csv_sha256": fingerprints,
    "preparation_settings": settings,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "timing_scope": "persistence_predict on existing validation array, including checks and copy"
}
with open(output_folder / "run_information.json", "w", encoding="utf-8") as file:
    json.dump(run_information, file, indent=2)

print("Saved baseline results to:", output_folder)
print("No final-test predictions have been made.")

## Review and GitHub checkpoint

Before moving to linear regression, explain what the validation MAE means in people and why a low training-period error is not sufficient. These results apply only to eligible examples for this space and period. They do not verify physical people counts, performance during missing-data periods, future room transfer, or HVAC energy savings.

The regression models will use the same prepared rows and `regression_metrics` function. They can use all five inputs; persistence deliberately uses only the current count. Comparing forecasts on the same rows keeps the benchmark fair.

Run the tests from the **project-root terminal**, not from a notebook cell:

```powershell
.\.venv\Scripts\python.exe -m pip install pytest
.\.venv\Scripts\python.exe -m pytest tests/test_evaluation.py -v --junitxml=docs/baseline_test_results.xml
```

The tests check implementation correctness with small synthetic examples, not predictive performance. They are the first part of the test suite, not complete tests of data preparation, leakage prevention or the later models.

**GitHub:** after running, reviewing and saving this notebook, commit the notebook, new `src` files, tests and your generated test report. Do not commit the prepared data or bulk outputs. No changes to Notebooks 01–02 are required.

### References
- MMA3001 Week 5.5, *Evaluating Regression Performance*: baseline, MAE/RMSE and residual interpretation.
- [Forecasting: Principles and Practice, simple methods](https://otexts.com/fpp3/simple-methods.html): naive/persistence forecast.
- [scikit-learn MAE](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html) and [RMSE](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html): metric definitions.
- [Python perf_counter](https://docs.python.org/3/library/time.html#time.perf_counter): elapsed timing.
- [pytest usage](https://docs.pytest.org/en/stable/how-to/usage.html): running implementation tests.

**AI contribution:** this initial notebook, source functions and tests were drafted with ChatGPT. User review, local execution, any corrections and modelling decisions should be reflected in the final AI-use reflection. An assistant-side execution is not a substitute for running and understanding the code locally.